# Video Summary as a JSON object



In [1]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import json

load_dotenv()

class Summary(BaseModel):
    """
    This is the template for the JSON object expected from the llm.
    """

    overview: str = Field(description="A high-level overview of the main topic and key points discussed.")
    key_takeaways: list[str] = Field(description="A list of the most important insights, facts, or steps.")
    conclusion: str = Field(
        description="A concise wrap-up of the video's overall message or next steps (if any), highlighting the core value or outcome."
    )


parser = JsonOutputParser(pydantic_object=Summary)

prompt = PromptTemplate.from_template(
    template="""
        Summarize the following video transcript clearly and concisely.

        {parser_instructions}

        Be concise but informative. Preserve the original intent and tone of the speaker.
        Don't include filler or irrelevant content.

        Transcript: {transcript}
    """
)

llm = ChatDeepSeek(model="deepseek-chat", temperature=0.1)


summarizer = prompt | llm | parser

In [2]:
with open("./video_text.txt", "r") as file:
    video_text = file.read()

response = summarizer.invoke(input={"parser_instructions": parser.get_format_instructions(), "transcript": video_text})
response

{'overview': 'A heated debate about whether women, particularly mothers, should prioritize staying home to raise children rather than pursuing careers. The male speaker argues that women have natural nurturing instincts that are wasted in corporate jobs, while the female speaker counters that women can balance both career and family responsibilities.',
 'key_takeaways': ['The male speaker believes women have natural instincts for child-rearing that are being wasted in corporate jobs',
  'He argues that corporate work is spiritually unfulfilling compared to raising children',
  'The female speaker maintains women can successfully balance careers and family',
  "Historical context is debated regarding women's awareness of the working world in previous generations",
  'Both speakers acknowledge tension between traditional gender roles and modern feminism'],
 'conclusion': "The debate remains unresolved, highlighting the ongoing tension between traditional views of motherhood and modern wo

In [3]:
type(response)

dict

Convert json to string

In [4]:
json_string = json.dumps(response)
json_string

'{"overview": "A heated debate about whether women, particularly mothers, should prioritize staying home to raise children rather than pursuing careers. The male speaker argues that women have natural nurturing instincts that are wasted in corporate jobs, while the female speaker counters that women can balance both career and family responsibilities.", "key_takeaways": ["The male speaker believes women have natural instincts for child-rearing that are being wasted in corporate jobs", "He argues that corporate work is spiritually unfulfilling compared to raising children", "The female speaker maintains women can successfully balance careers and family", "Historical context is debated regarding women\'s awareness of the working world in previous generations", "Both speakers acknowledge tension between traditional gender roles and modern feminism"], "conclusion": "The debate remains unresolved, highlighting the ongoing tension between traditional views of motherhood and modern women\'s d